# 04. Evaluar y visualizar modelos

Este notebook compara las predicciones locales generadas por el notebook 03 contra los `reference_claim` anotados de `test.csv`. METEOR, ROUGE-L y BERTScore miden similitud con la referencia; no validan la veracidad médica de una noticia.

## Requisitos

Ejecuta antes el notebook 03. Para METEOR y BERTScore, instala `pip install -e '.[analysis,models,dev]'` y descarga una vez los recursos de NLTK con `python -m nltk.downloader wordnet omw-1.4`.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').is_file() and (ROOT.parent / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Abre el notebook desde la raíz del repositorio o desde notebooks/.')

FINAL_DIR = ROOT / 'artifacts' / 'sem1' / 'final'
RUN_NAME = 'current'  # debe coincidir con el notebook 03
RUN_DIR = ROOT / 'artifacts' / 'sem1' / 'runs' / RUN_NAME
PREDICTIONS_DIR = RUN_DIR / 'predictions'
EVALUATION_DIR = RUN_DIR / 'evaluation'
MODEL_ORDER = ['t5s_spanish', 'barto_spanish', 'mbart50_es', 'umt5_base_es']

print({'run_dir': str(RUN_DIR), 'predictions_dir': str(PREDICTIONS_DIR)})

In [ ]:
import json

import pandas as pd

from claim_extraction.workflow import evaluate_predictions, utc_now, validate_sem1_splits

SPLIT_PATHS = {name: FINAL_DIR / f'{name}.csv' for name in ('train', 'validation', 'test')}
splits = validate_sem1_splits(SPLIT_PATHS)
test = splits['test']

prediction_paths = [PREDICTIONS_DIR / f'{model_id}.csv' for model_id in MODEL_ORDER]
missing = [path for path in prediction_paths if not path.is_file()]
if missing:
    raise FileNotFoundError('Faltan predicciones: ' + ', '.join(str(path) for path in missing))

prediction_frames = [pd.read_csv(path, dtype=str, keep_default_na=False) for path in prediction_paths]
predictions = pd.concat(prediction_frames, ignore_index=True)
expected_test_ids = set(test['record_id'])
for model_id, group in predictions.groupby('model_id', sort=True):
    if set(group['record_id']) != expected_test_ids or len(group) != len(test):
        raise ValueError(f'{model_id} no cubre exactamente los casos de test.csv.')

print({'test_rows': len(test), 'prediction_rows': len(predictions), 'models': sorted(predictions['model_id'].unique())})

## Calcular métricas

Cada salida se guarda localmente bajo la carpeta de la corrida. Los cuerpos de noticias, claims de referencia y predicciones individuales permanecen ignorados por Git.

In [ ]:
summary, per_prediction = evaluate_predictions(predictions, test)
summary = summary.set_index('model_id').reindex(MODEL_ORDER).reset_index()
if summary[['meteor', 'rouge_l_f1', 'bertscore_f1']].isna().any().any():
    raise AssertionError('No se calcularon las tres métricas de referencia para todos los modelos.')

EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(EVALUATION_DIR / 'summary.csv', index=False, encoding='utf-8')
per_prediction.to_csv(EVALUATION_DIR / 'per_prediction.csv', index=False, encoding='utf-8')
for model_id, group in per_prediction.groupby('model_id', sort=True):
    model_dir = EVALUATION_DIR / model_id
    model_dir.mkdir(parents=True, exist_ok=True)
    summary.loc[summary['model_id'].eq(model_id)].to_csv(model_dir / 'summary.csv', index=False, encoding='utf-8')
    group.to_csv(model_dir / 'per_prediction.csv', index=False, encoding='utf-8')

status_path = RUN_DIR / 'run_status.json'
status = json.loads(status_path.read_text(encoding='utf-8')) if status_path.is_file() else {}
status['status'] = 'completed'
status['evaluated_at_utc'] = utc_now()
status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding='utf-8')

summary[['model_display_name', 'prediction_n', 'reference_n', 'meteor', 'rouge_l_f1', 'bertscore_f1']]

## Tablas y figura locales

Estas tablas sirven para redactar resultados. No publiques textos, anotaciones ni predicciones individuales: solo los agregados revisados.

In [ ]:
import matplotlib.pyplot as plt

TABLES_DIR = EVALUATION_DIR / 'tables'
FIGURES_DIR = EVALUATION_DIR / 'figures'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

split_table = pd.DataFrame([
    {'split': name, 'filas': len(frame)} for name, frame in splits.items()
])
metrics_table = summary[[
    'model_id', 'model_display_name', 'prediction_n', 'reference_n',
    'meteor', 'rouge_l_f1', 'bertscore_f1',
]].copy()
diagnostics_table = summary[[
    'model_id', 'empty_rate', 'shorter_than_5_tokens_rate',
    'mean_prediction_tokens_operational', 'duplicate_output_rate',
]].copy()
split_table.to_csv(TABLES_DIR / '01_distribucion_particiones.csv', index=False, encoding='utf-8')
metrics_table.to_csv(TABLES_DIR / '02_metricas_principales.csv', index=False, encoding='utf-8')
diagnostics_table.to_csv(TABLES_DIR / '03_diagnosticos_generacion.csv', index=False, encoding='utf-8')

metric_columns = {'meteor': 'METEOR', 'rouge_l_f1': 'ROUGE-L F1', 'bertscore_f1': 'BERTScore F1'}
figure, axes = plt.subplots(1, 3, figsize=(12, 3.8), layout='constrained')
for axis, (column, label) in zip(axes, metric_columns.items(), strict=True):
    bars = axis.bar(metrics_table['model_display_name'], metrics_table[column], color='#277f8e')
    axis.bar_label(bars, labels=[f'{value:.3f}' for value in metrics_table[column]], padding=3, fontsize=9)
    axis.set(title=label, ylabel='Puntaje medio', ylim=(0, 1))
    axis.tick_params(axis='x', rotation=20)
    axis.grid(axis='y', alpha=.2)
figure.savefig(FIGURES_DIR / '01_comparacion_metricas.png', dpi=180, bbox_inches='tight')
plt.show()

print(f'Resultados locales: {EVALUATION_DIR.relative_to(ROOT)}')